# Lab 03.2 — Spark su Hadoop YARN

**YARN** (Yet Another Resource Negotiator) è il gestore di risorse di Hadoop: lo stesso cluster può eseguire Spark, MapReduce, Hive, ...

Con `master("yarn")`:

1. il driver (questo notebook) chiede al **ResourceManager** di avviare un'applicazione;
2. su un **NodeManager** parte l'**ApplicationMaster** di Spark, che negozia i container per gli executor;
3. gli executor girano in **container YARN** con memoria e CPU assegnate.

> Questo notebook richiede la modalità YARN: `./lab.sh up 03 --yarn`
> ed i file creati dal notebook `01_hdfs_e_spark` (eseguilo prima, anche in modalità YARN).

UI: ResourceManager http://localhost:8088 · NodeManager http://localhost:8042

In [ ]:
import os, socket, requests
from sessione import crea_sessione

assert os.environ.get("SPARK_MASTER") == "yarn", "Avvia il laboratorio con: ./lab.sh up 03 --yarn"

spark = crea_sessione("Lab03-YARN")
sc = spark.sparkContext
print("Master        :", sc.master)
print("Application ID:", sc.applicationId, " (formato YARN: application_<timestamp>_<n>)")

## 1. Configurazione usata
Il driver trova il ResourceManager leggendo `yarn-site.xml` dalla cartella indicata da `HADOOP_CONF_DIR`.

In [ ]:
print("HADOOP_CONF_DIR:", os.environ["HADOOP_CONF_DIR"])
hconf = sc._jsc.hadoopConfiguration()
for k in ["yarn.resourcemanager.hostname", "yarn.nodemanager.resource.memory-mb", "fs.defaultFS"]:
    print(f"  {k} = {hconf.get(k)}")
for k in ["spark.executor.instances", "spark.executor.memory", "spark.yarn.am.memory", "spark.yarn.jars"]:
    print(f"  {k} = {sc.getConf().get(k)}")

## 2. Dove girano i task?
Gli executor sono processi nei container YARN del NodeManager.

In [ ]:
info = sc.parallelize(range(8), 8).map(lambda _: (socket.gethostname(), os.environ.get("CONTAINER_ID", "?"))).distinct().collect()
for host, container in sorted(info):
    print(f"host {host:12s} container {container}")

## 3. Il ResourceManager vede l'applicazione
Interroghiamo la REST API di YARN: stato, coda, risorse allocate.

In [ ]:
rm = "http://resourcemanager:8088/ws/v1/cluster"
app = requests.get(f"{rm}/apps/{sc.applicationId}").json()["app"]
print({k: app[k] for k in ("name", "queue", "state", "allocatedMB", "allocatedVCores", "runningContainers")})

metriche = requests.get(f"{rm}/metrics").json()["clusterMetrics"]
print(f"Cluster: {metriche['allocatedMB']} / {metriche['totalMB']} MB allocati, "
      f"{metriche['allocatedVirtualCores']} / {metriche['totalVirtualCores']} vcore")

## 4. Un job MapReduce su YARN che legge da HDFS
Usiamo il file caricato nel notebook precedente (`/dati/alice.txt`).

In [ ]:
import re
from pyspark.sql import functions as F

top = (sc.textFile("/dati/alice.txt")
         .flatMap(lambda l: re.findall(r"[a-z]+", l.lower()))
         .filter(lambda w: len(w) > 4)
         .map(lambda w: (w, 1))
         .reduceByKey(lambda a, b: a + b)
         .takeOrdered(10, key=lambda kv: -kv[1]))
print(top)

spark.read.parquet("/dati/vendite").groupBy("regione").agg(F.round(F.avg("importo"), 2).alias("media")).show()

## 5. Chiusura
Con `spark.stop()` l'applicazione termina e YARN libera i container (stato `FINISHED` nella UI del ResourceManager).

In [ ]:
app_id = sc.applicationId
spark.stop()
import time; time.sleep(3)
print(app_id, "→", requests.get(f"{rm}/apps/{app_id}").json()["app"]["state"])